# Anchorage Harbor: building the dataset

From eHydro surveys to a stack of depth images, ready for the time-series step.

1. Setup
2. List all Anchorage surveys from the eHydro catalogue
3. Read one survey into a DataFrame (`x, y, depth`)
4. Clean it
5. Put the points on a 10 m grid
6. Process every survey (year by year, saved to Drive)
7. Quality checks
8. Turn the table into images

**Why 10 m?** Surveys from 2014–2017 only have points every ~30 ft (~9 m); from 2018 there are points every 3 ft.
A 10 m cell gets data in every year, so the whole 2014–2026 record is on the same grid.

## 1. Setup

In [ ]:
!pip install -q remotezip

import io
import os
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from remotezip import RemoteZip

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/ehydro/anchorage/dataset'
except ImportError:              # not in Colab: save in data/dataset next to the notebook
    SAVE_DIR = 'data/dataset'

os.makedirs(SAVE_DIR, exist_ok=True)
print('Saving to', SAVE_DIR)

## 2. List all Anchorage surveys

One request to the eHydro catalogue gives every condition survey (`CS`) of Anchorage Harbor with its date and zip link.

The file name tells the survey type: `IS` = intermediate survey (summer, during the dredging season),
`WS` = winter survey (smaller and sparser).

In [ ]:
CATALOG_URL = ('https://services7.arcgis.com/n1YM8pTrFmm7L4hs/arcgis/rest/services/'
               'eHydro_Survey_Data/FeatureServer/0/query')

params = {
    'where': "channelareaidfk = 'CEPOA_AK_16_ANC' AND surveytype = 'CS'",
    'outFields': 'surveydatestart,sourcedatalocation',
    'returnGeometry': 'false',
    'f': 'json',
}
records = [f['attributes'] for f in requests.get(CATALOG_URL, params=params).json()['features']]

surveys = pd.DataFrame(records)
surveys['date'] = pd.to_datetime(surveys['surveydatestart'], unit='ms').dt.normalize()
surveys['url'] = surveys['sourcedatalocation']
surveys['name'] = surveys['url'].str.split('/').str[-1].str.replace('.ZIP', '', case=False)
surveys['kind'] = surveys['name'].str.extract(r'_CS_([A-Za-z]+)')[0].str.upper().fillna('other')
surveys = surveys[['date', 'name', 'kind', 'url']].sort_values('date').reset_index(drop=True)
surveys['survey_id'] = surveys.index

print(len(surveys), 'surveys,', surveys.date.min().date(), 'to', surveys.date.max().date())
pd.crosstab(surveys.date.dt.year, surveys.kind)

## 3. Read one survey into a DataFrame

We download only the depth file from the zip (not the PDF):
the `_FULL.XYZ` file if there is one, otherwise the biggest `.XYZ`.

Every Anchorage depth file has the same layout: some header lines containing `==`
(survey company, notes, datum), then one point per line: `x,y,elevation`
(comma-separated in almost all files; one file uses spaces, so we accept both).

- x, y are US survey feet (Alaska State Plane zone 4) → converted to metres
- elevation is negative feet below MLLW (the low-tide level) → flipped to positive depth in feet

In [ ]:
FEET_TO_M = 1200 / 3937          # one US survey foot in metres

def read_points(url):
    """Download the depth file of one survey. Returns a DataFrame with x, y (m) and depth (ft), and the file name."""
    with RemoteZip(url, support_suffix_range=False) as z:
        xyz_files = [i for i in z.infolist() if '/' not in i.filename and i.filename.upper().endswith('.XYZ')]
        full = [i for i in xyz_files if i.filename.upper().endswith('_FULL.XYZ')]
        chosen = full[0] if full else max(xyz_files, key=lambda i: i.file_size)
        text = z.read(chosen.filename).decode('latin-1')

    data_lines = [line for line in text.splitlines() if '==' not in line and line.strip()]
    data = '\n'.join(data_lines).replace(',', ' ')        # most files use commas, a few use spaces
    points = pd.read_csv(io.StringIO(data), sep=r'\s+', header=None, names=['x', 'y', 'elevation'])

    points['x'] = points['x'] * FEET_TO_M
    points['y'] = points['y'] * FEET_TO_M
    points['depth'] = -points['elevation']
    return points[['x', 'y', 'depth']], chosen.filename

In [ ]:
example = surveys[surveys.date == '2025-06-30'].iloc[0]
points, file_name = read_points(example.url)

print(example['name'], '->', file_name)
print(f'{len(points):,} points')
points.describe().round(2)

## 4. Clean it

Kept simple on purpose. Per survey we:
- drop exact duplicate points
- drop depths outside a plausible range (more than 30 ft above low tide, or deeper than 120 ft).
  Real values here are about -5 to 75 ft, so this only removes obvious errors.
- drop points outside a box around the harbor. All surveys fall within x 504,740–506,410 m and
  y 804,980–808,290 m; the box adds ~500 m margin. (One winter survey, 2023-11-15, has 7 points with
  broken coordinates up to 770 km away.)

We do **not** fill gaps: an area a survey didn't cover simply has no points.

In [ ]:
MIN_DEPTH, MAX_DEPTH = -30, 120      # ft
X_MIN, X_MAX = 504_000, 507_000      # m, harbor box
Y_MIN, Y_MAX = 804_500, 808_800      # m

def clean(points):
    """Drop duplicate points, impossible depths and points outside the harbor.
    Returns the cleaned points and how many were dropped."""
    n_before = len(points)
    points = points.drop_duplicates()
    points = points[(points.depth >= MIN_DEPTH) & (points.depth <= MAX_DEPTH)]
    points = points[points.x.between(X_MIN, X_MAX) & points.y.between(Y_MIN, Y_MAX)]
    return points, n_before - len(points)

points, n_dropped = clean(points)
print('dropped', n_dropped, 'points')

## 5. Put the points on a 10 m grid

Each point gets a grid cell: `col = x // 10`, `row = y // 10` (in metres).
Using absolute coordinates means every survey lands on the same grid automatically.

For each cell we keep the mean depth and how many points it was built from.
The result is a small table: one row per cell (~7,000 for a full summer survey, instead of ~800,000 points).

In [ ]:
CELL = 10        # metres

def to_cells(points):
    """Average the points in each 10 m cell."""
    points = points.assign(row=(points.y // CELL).astype(int), col=(points.x // CELL).astype(int))
    return points.groupby(['row', 'col']).depth.agg(depth='mean', n_points='size').reset_index()

cells = to_cells(points)
print(len(cells), 'cells')

plt.figure(figsize=(5, 7))
plt.scatter(cells.col, cells.row, c=cells.depth, s=2, marker='s', cmap='viridis_r')
plt.colorbar(label='depth (ft below MLLW)')
plt.gca().set_aspect('equal')
plt.title(f"{example['name']}: 10 m cells")
plt.show()

## 6. Process every survey

Same three steps (read, clean, grid) for all ~880 surveys. Downloads run 8 at a time to save time.

Results are saved to Drive **one year at a time**. If Colab disconnects, rerun this cell:
years already saved are skipped. Expect roughly 15–30 minutes for everything.

In [ ]:
def process(survey):
    """Read, clean and grid one survey. Returns its cells and a line of info for the log."""
    try:
        points, file_name = read_points(survey.url)
        n_points = len(points)
        points, n_dropped = clean(points)
        cells = to_cells(points)
        cells['survey_id'] = survey.survey_id
        info = {'survey_id': survey.survey_id, 'file': file_name, 'n_points': n_points,
                'n_dropped': n_dropped, 'n_cells': len(cells), 'error': None}
    except Exception as error:
        cells = None
        info = {'survey_id': survey.survey_id, 'error': str(error)}
    return cells, info

In [ ]:
YEARS = sorted(surveys.date.dt.year.unique())

for year in YEARS:
    cells_path = f'{SAVE_DIR}/cells_{year}.parquet'
    if os.path.exists(cells_path):
        print(year, 'already done')
        continue

    year_surveys = [s for _, s in surveys[surveys.date.dt.year == year].iterrows()]
    with ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(process, year_surveys))

    year_cells = pd.concat([c for c, _ in results if c is not None])
    year_info = pd.DataFrame([info for _, info in results])
    year_cells.to_parquet(cells_path)
    year_info.to_csv(f'{SAVE_DIR}/info_{year}.csv', index=False)
    print(year, f'{len(year_surveys)} surveys, {len(year_cells):,} cells, errors: {year_info.error.notna().sum()}')

## 7. Quality checks

Load everything back into two tables:
- `table`: one row per (survey, cell) with the depth: the main dataset
- `info`: one row per survey: which file was used, how many points, how many cells, any error

In [ ]:
table = pd.concat([pd.read_parquet(f'{SAVE_DIR}/cells_{y}.parquet') for y in YEARS], ignore_index=True)
info = pd.concat([pd.read_csv(f'{SAVE_DIR}/info_{y}.csv') for y in YEARS], ignore_index=True)
info = surveys.merge(info, on='survey_id')
surveys.to_csv(f'{SAVE_DIR}/surveys.csv', index=False)

print(f'{len(table):,} rows in the table,', table.survey_id.nunique(), 'surveys')
print('surveys with an error:', info.error.notna().sum())
info[info.error.notna()][['date', 'name', 'error']]

Which depth file did each year have? Expect `_FULL.XYZ` from 2018 and only the thinned file before.

In [ ]:
info['file_type'] = np.where(info.file.str.upper().str.endswith('_FULL.XYZ'), 'FULL', 'thinned')
pd.crosstab(info.date.dt.year, info.file_type)

How much of the harbor does each survey cover, and how many points went into each cell?
Expect full summer (`IS`) surveys of ~7,000 cells and smaller winter (`WS`) surveys.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for kind, group in info.groupby('kind'):
    ax1.plot(group.date, group.n_cells, 'o', ms=3, label=kind)
    ax2.plot(group.date, group.n_points / group.n_cells, 'o', ms=3, label=kind)
ax1.set_ylabel('cells with data')
ax2.set_ylabel('points per cell')
ax2.set_yscale('log')
ax1.legend()
plt.show()

Several surveys on the same day? We keep them all for now and decide in the time-series step.

In [ ]:
same_day = info[info.duplicated('date', keep=False)]
print(len(same_day), 'surveys share a date with another survey')
same_day[['date', 'name', 'kind', 'n_cells']].head(10)

## 8. Turn the table into images

Every cell has an absolute `row` and `col`. We shift them so the smallest is 0,
then place each survey's depths into a 2-D array. Cells without data stay `NaN`.

Result: `images[survey, row, col]`, with `surveys` telling the date of each image.

In [ ]:
row0, col0 = table.row.min(), table.col.min()
H = table.row.max() - row0 + 1
W = table.col.max() - col0 + 1
print(f'image size: {H} x {W} cells of {CELL} m')

images = np.full((len(surveys), H, W), np.nan, dtype=np.float32)
images[table.survey_id, table.row - row0, table.col - col0] = table.depth

print('images shape (surveys, rows, cols):', images.shape)

In [ ]:
# the first July survey of a few years, plus the latest winter survey
july = [surveys[(surveys.date.dt.year == y) & (surveys.date.dt.month == 7)].head(1) for y in [2014, 2017, 2020, 2023, 2026]]
examples = pd.concat(july + [surveys[surveys.kind == 'WS'].tail(1)])

fig, axes = plt.subplots(1, len(examples), figsize=(3 * len(examples), 5))
for ax, (_, s) in zip(axes, examples.iterrows()):
    ax.imshow(images[s.survey_id], origin='lower', cmap='viridis_r', vmin=0, vmax=70)
    ax.set_title(f'{s.date.date()} {s.kind}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

Save the images with everything needed to use them later: dates, survey ids, and where the grid sits.

In [ ]:
np.savez_compressed(f'{SAVE_DIR}/images_10m.npz', images=images, survey_id=surveys.survey_id.values,
                    dates=surveys.date.values.astype('datetime64[D]'), row0=row0, col0=col0, cell=CELL)
info.to_csv(f'{SAVE_DIR}/info.csv', index=False)

print('saved', f'{SAVE_DIR}/images_10m.npz', f'({os.path.getsize(f"{SAVE_DIR}/images_10m.npz") / 1e6:.0f} MB)')